*Notebook 04 in `notebooks/arcs/`. Parameters (pre-registered): `analysis/arcs/config.yaml`; decisions and run notes: `analysis/arcs/LOG.md`. The static page `site/arcs/index.html` is built from the same saved outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/arcs -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.arcs import s04_validation as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 4 · Human validation sheets

**What we do.** We draw two sheets for people to fill in. The code never fills them in.
- **50 scenes** from the primary sample, drawn at random in proportion to each genre bin. Each row carries the
  instrument scene scores and empty columns for two raters: a label (positive / negative / neutral / mixed)
  and, where a tool would go wrong, an error category (negation, sarcasm, event-not-tone, tokenisation/markup,
  other).
- **25 scripts** drawn at random from all 1,087. Each row has empty columns: is the markup right, is it a final
  script, a draft or a transcript, and does the MovieSum script belong to the IMDb film we joined it to.

**Labelling is blind.** Raters read a private preview with the scene text and no score, so a number can't
anchor their judgement. The preview and the IMDb titles stay in `private/` and are never committed.

**Scoring is fixed in advance** (`validation.scoring` in the config):
- Every instrument's scene score maps to a label with the same ±0.05 band.
- Accuracy is reported with "mixed" counted as a miss, and on non-mixed scenes only.
- Cohen's kappa measures agreement between the two raters.
- The error table counts each category, overall and among the primary instrument's misses.

**What this validation can and cannot show.** 50 scenes is a small sample: an accuracy of 70% has a 95%
interval of roughly ±13 points. That is enough to show whether an instrument is usable or badly off, and not
enough to rank two instruments that are close. A scene label is also coarser than the arc, which averages many
scenes.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

**Table.** Sheet status. Re-running this step refreshes only the score columns and recomputes these numbers. It never redraws the sample or touches a human column.

In [ ]:
display(step.tables(res)["status"])

,item,value
0,scenes_in_sheet,50
1,scenes_labelled_rater1,0
2,scenes_labelled_rater2,0
3,scenes_resolved,0
4,scenes_unresolved,0
5,scripts_in_sheet,25
6,scripts_checked,0
7,status,not yet validated


**Table.** Scenes drawn per genre bin, against each bin's share of all scenes on the primary timeline.

In [ ]:
display(step.tables(res)["allocation"])

,genre_bin,scenes drawn,share of primary scenes
0,Action,4,9.1%
1,Adventure,6,12.6%
2,Comedy,9,18.0%
3,Drama,13,25.4%
4,Horror,5,11.0%
5,Other,1,0.7%
6,Sci-Fi,5,10.1%
7,Thriller,7,13.2%


**Table.** The scene sheet as committed: MovieSum name, scene index, size, instrument scores and the empty rater columns. The contextual columns stay empty until the HPC job returns.

In [ ]:
display(step.tables(res)["scene_sheet"])

,sheet_id,movie_name,scene,genre_bin,sentences,n_dialogue,n_action,vader_scene,sst2_scene,roberta3_scene,sst2_scene_context,roberta3_scene_context,rater1_label,rater1_error,rater2_label,rater2_error,adjudicated_label
0,S01,I Am Number Four_2011,98,Sci-Fi,4,0,4,-0.1562,,,,,,,,,
1,S02,Ferris Bueller's Day Off_1986,186,Comedy,3,0,3,0.1061,,,,,,,,,
2,S03,The Life of David Gale_2003,94,Drama,6,0,6,0.0000,,,,,,,,,
3,S04,Capote_2005,65,Drama,5,0,5,0.0000,,,,,,,,,
4,S05,Pet Sematary_1989,517,Horror,5,0,5,0.0000,,,,,,,,,
5,S06,The Bucket List_2007,116,Adventure,11,5,6,0.0692,,,,,,,,,
6,S07,The Naked Gun: From the Files of Police Squad!_1988,91,Comedy,29,21,8,0.0218,,,,,,,,,
7,S08,Rango_2011,60,Adventure,46,36,10,0.0435,,,,,,,,,
8,S09,The Truman Show_1998,134,Comedy,8,0,8,-0.0146,,,,,,,,,
9,S10,The King's Speech_2010,35,Drama,2,0,2,0.0000,,,,,,,,,


**Table.** The script sheet as committed, with the Step 2 flags beside the empty check columns.

In [ ]:
display(step.tables(res)["script_sheet"])

,sheet_id,movie_name,genre_bin,n_scene_tags,sentences_both,action_share,pretokenised,possible_transcript,numbered_scenes,in_primary,markup_ok,kind,id_mapping_ok,rater
0,P01,Angels & Demons_2009,Thriller,152.0,2710,0.685,True,False,False,False,,,,
1,P02,Batman Begins_2005,Other,325.0,3414,0.650,True,False,False,True,,,,
2,P03,The Ugly Truth_2009,Comedy,133.0,2527,0.363,True,False,False,True,,,,
3,P04,Donnie Brasco_1997,Drama,118.0,3209,0.349,False,False,False,True,,,,
4,P05,Dunkirk_2017,Action,198.0,1752,0.911,False,False,False,True,,,,
5,P06,The Avengers_2012,Sci-Fi,221.0,2787,0.557,False,False,False,True,,,,
6,P07,Life of Pi_2012,Adventure,123.0,1699,0.645,True,False,False,True,,,,
7,P08,Fences_2016,Drama,40.0,3514,0.975,False,False,False,True,,,,
8,P09,Sing_2016,Comedy,153.0,2932,0.514,False,False,False,False,,,,
9,P10,Terminator Genisys_2015,Sci-Fi,117.0,2940,0.688,False,False,False,False,,,,


**Private check.** where the private rater previews were written (text stays on disk).

In [ ]:
display(step.private(res)["preview"])   # output cleared before commit

**Table.** Per-instrument accuracy against the human reference label.

In [ ]:
display(step.tables(res)["accuracy"])

,result
0,NOT YET VALIDATED: the sheets have not been filled


**Table.** Error categories given by the raters.

In [ ]:
display(step.tables(res)["errors"])

,result
0,NOT YET VALIDATED: the sheets have not been filled


**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

The scene sheet holds 50 scenes from the primary sample, drawn in proportion to each genre bin (8 bins, 1 to 13 scenes each); the script sheet holds 25 scripts. Scenes range from 1 to 66 sentences (median 8). **NOT YET VALIDATED: the sheets have not been filled.** Every human column is empty. Until it is filled, no instrument has been checked against a human reader, and every result built on the sentiment scores carries that caveat.